# Movie features

Turn `movies_cleaned.csv` into the model-ready `movies_features.csv`, using the checks in `04_eda_movies.ipynb`. Row filters and encodings are applied here. The people columns are the director name and the first `K` billed actors. Track-record scores are not written: they use other movies' ratings and are computed after the train/test split.


## 1. Load the cleaned table

List columns are JSON text. `K` below is the number of billed actors stored with each movie.


In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path("..")
PROCESSED = ROOT / "data" / "processed"
MIN_MONEY = 1_000
NETFLIX_END_YEAR = 2005
US = "United States of America"
TOP_COMPANIES = 15
K = 10


In [2]:
def parse_list(val):
    if pd.isna(val) or val == "":
        return []
    return json.loads(val)


def slugify(name):
    return "".join(ch if ch.isalnum() else "_" for ch in name.lower()).strip("_")


movies = pd.read_csv(PROCESSED / "movies_cleaned.csv")
for col in ["genre_names", "company_names", "country_names", "language_names", "cast_names"]:
    movies[col] = movies[col].map(parse_list)
print(movies.shape)
print("K =", K)


(1806, 21)
K = 10


## 2. Drop impossible budget, revenue, or runtime

TMDB stores unknown budget, revenue, and runtime as 0, and a few other rows use amounts under $1,000. Both break `log(revenue / budget)`. Drop a movie if either amount is under $1,000 or runtime is missing or 0.


In [3]:
n_before = len(movies)
bad = (
    (movies["budget"] < MIN_MONEY)
    | (movies["revenue"] < MIN_MONEY)
    | movies["runtime"].isna()
    | (movies["runtime"] == 0)
)
movies = movies.loc[~bad].reset_index(drop=True)
print(f"dropped {n_before - len(movies)}; remaining {len(movies)}")


dropped 492; remaining 1314


## 3. Numeric features

Keep raw runtime. Age is `log1p(2005 - year)`. After the money filter, `budget_log` is `log(budget)` and `log_roi` is `log(revenue) - log(budget)`. Raw year, age, budget, and revenue stay out of the feature file because they repeat these columns.


In [4]:
movies["movie_age_log"] = np.log1p(NETFLIX_END_YEAR - movies["year"])
movies["budget_log"] = np.log(movies["budget"])
movies["log_roi"] = np.log(movies["revenue"]) - np.log(movies["budget"])
movies[["runtime", "movie_age_log", "budget_log", "log_roi"]].describe().round(2)


,runtime,movie_age_log,budget_log,log_roi
count,1314.00,1314.00,1314.00,1314.00
mean,112.79,2.06,16.77,0.83
std,21.76,0.94,1.38,1.37
min,69.00,0.00,8.85,-6.48
25%,97.00,1.39,16.12,0.07
50%,108.00,2.08,17.03,0.85
75%,124.00,2.71,17.73,1.56
max,219.00,4.50,19.11,8.33


## 4. Genres

One column per genre. Drop `Foreign`. Keep `Documentary`.


In [5]:
genres = sorted({name for names in movies["genre_names"] for name in names} - {"Foreign"})
genre_df = pd.DataFrame(
    {
        f"genre_{slugify(name)}": movies["genre_names"].map(lambda names, g=name: int(g in names))
        for name in genres
    }
)
print(f"{genre_df.shape[1]} genre columns; movies with none: {int((genre_df.sum(axis=1) == 0).sum())}")


18 genre columns; movies with none: 0


## 5. Production companies

Merge studio aliases, then keep the 15 most frequent names and an other flag.


In [6]:
COMPANY_ALIASES = {
    "Columbia Pictures Corporation": "Columbia Pictures",
    "Paramount": "Paramount Pictures",
    "Warner Bros. Pictures": "Warner Bros.",
    "Universal Studios": "Universal Pictures",
    "Universal Pictures Corporation": "Universal Pictures",
    "Miramax": "Miramax Films",
    "Walt Disney": "Walt Disney Pictures",
    "Walt Disney Productions": "Walt Disney Pictures",
    "DreamWorks": "DreamWorks SKG",
    "DreamWorks Pictures": "DreamWorks SKG",
    "Polygram Filmed Entertainment": "PolyGram Filmed Entertainment",
    "Lions Gate": "Lions Gate Films",
    "Malpaso Company": "Malpaso Productions",
    "Cruise-Wagner Productions": "Cruise/Wagner Productions",
    "Mandalay Pictures": "Mandalay Entertainment",
    "Zanuck Company, The": "The Zanuck Company",
    "Kennedy/Marshall Company, The": "The Kennedy/Marshall Company",
}
movies["company_names"] = movies["company_names"].map(
    lambda names: list(dict.fromkeys(COMPANY_ALIASES.get(name, name) for name in names))
)
company_counts = movies["company_names"].explode().dropna().value_counts()
top_companies = list(company_counts.index[:TOP_COMPANIES])
company_df = pd.DataFrame(
    {
        f"company_{slugify(name)}": movies["company_names"].map(lambda names, g=name: int(g in names))
        for name in top_companies
    }
)
company_df["company_other"] = movies["company_names"].map(
    lambda names: int(len(names) == 0 or any(name not in top_companies for name in names))
)
print(company_df.sum().sort_values(ascending=False).head(8).to_string())


company_other                                     1092
company_paramount_pictures                         140
company_warner_bros                                136
company_columbia_pictures                          136
company_universal_pictures                         125
company_twentieth_century_fox_film_corporation      94
company_touchstone_pictures                         65
company_new_line_cinema                             62


## 6. Production countries

Three groups: US only, a US co-production, and non-US. A movie with no listed country is 0 on all three.


In [7]:
country_df = pd.DataFrame(
    {
        "us_only": movies["country_names"].map(lambda names: int(names == [US])),
        "us_coproduction": movies["country_names"].map(lambda names: int(US in names and len(names) > 1)),
        "non_us": movies["country_names"].map(lambda names: int(len(names) > 0 and US not in names)),
    }
)
print("no country:", int((country_df.sum(axis=1) == 0).sum()))
print(country_df.sum().to_string())


no country: 5
us_only            944
us_coproduction    279
non_us              86


## 7. Language

`lang_en` marks an English original language. `is_multilingual` marks two or more spoken languages.


In [8]:
language_df = pd.DataFrame(
    {
        "lang_en": (movies["original_language"] == "en").astype(int),
        "is_multilingual": movies["language_names"].map(lambda names: int(len(names) > 1)),
    }
)
print(language_df.mean().round(3).to_string())


lang_en            0.976
is_multilingual    0.345


## 8. People columns

`K` is the number of billed actors kept, in billing order. The file stores the director name and those actor names. The track-record score from the EDA uses other movies' ratings, so it is computed after the train/test split.


In [9]:
people_df = pd.DataFrame(
    {
        "director": movies["director"],
        f"cast_top{K}": movies["cast_names"].map(lambda names: json.dumps(names[:K])),
    }
)
print(list(people_df.columns))
people_df.head(2)


['director', 'cast_top10']


,director,cast_top10
0,Nancy Meyers,"[""Jack Nicholson"", ""Diane Keaton"", ""Keanu Reev..."
1,William Friedkin,"[""David Caruso"", ""Linda Fiorentino"", ""Chazz Pa..."


## 9. Assemble and save

Keep ids, title, the numeric columns, the encodings, and the people columns. Do not write year, decade dummies, scaled copies, or anything computed from ratings.


In [10]:
base = movies[
    ["netflix_movie_id", "tmdb_id", "title", "runtime", "movie_age_log", "budget_log", "log_roi"]
].reset_index(drop=True)
features = pd.concat(
    [base, genre_df, company_df, country_df, language_df, people_df.reset_index(drop=True)],
    axis=1,
)
assert features["netflix_movie_id"].is_unique
model_cols = [col for col in features.columns if col not in {"director"} and not col.startswith("cast_top")]
assert features[model_cols].notna().all().all()

out_path = PROCESSED / "movies_features.csv"
features.to_csv(out_path, index=False)
print(f"saved {out_path} ({features.shape[0]} rows x {features.shape[1]} cols)")
print(f"people columns use the first {K} billed actors")
features.head(2)


saved ..\data\processed\movies_features.csv (1314 rows x 48 cols)
people columns use the first 10 billed actors


,netflix_movie_id,tmdb_id,title,runtime,movie_age_log,budget_log,log_roi,genre_action,genre_adventure,genre_animation,...,company_tristar_pictures,company_amblin_entertainment,company_other,us_only,us_coproduction,non_us,lang_en,is_multilingual,director,cast_top10
0,30,6964,Something's Gotta Give,128.0,1.098612,18.197537,1.204206,0,0,0,...,0,0,1,1,0,0,1,1,Nancy Meyers,"[""Jack Nicholson"", ""Diane Keaton"", ""Keanu Reev..."
1,55,11863,Jade,95.0,2.397895,17.727534,-1.624388,1,0,0,...,0,0,0,1,0,0,1,0,William Friedkin,"[""David Caruso"", ""Linda Fiorentino"", ""Chazz Pa..."
